# Payments Operations Analysis

**Section 4e Demo** — Notebooks combine SQL and Python in a single governed environment.
This notebook queries payment data, analyses failure patterns, and produces operational charts.

Run inside Snowflake — no data leaves, uses your warehouse compute.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import matplotlib.pyplot as plt
import streamlit as st

session = get_active_session()
session.sql('USE DATABASE BARCLAYS_DEMO').collect()
session.sql('USE SCHEMA RAW').collect()
session.sql('USE WAREHOUSE BARCLAYS_DEMO_WH').collect()
print('Connected to BARCLAYS_DEMO.RAW')

## 1. Payment Volume by Type and Status
Query the payments table and bring the results into Pandas.

In [ ]:
df_volume = session.sql("""
    SELECT 
        PAYMENT_TYPE,
        STATUS,
        COUNT(*) AS TXN_COUNT,
        ROUND(SUM(AMOUNT), 0) AS TOTAL_VALUE,
        ROUND(AVG(PROCESSING_TIME_MS), 0) AS AVG_PROCESSING_MS
    FROM PAYMENTS
    GROUP BY PAYMENT_TYPE, STATUS
    ORDER BY PAYMENT_TYPE, TXN_COUNT DESC
""").to_pandas()

st.dataframe(df_volume)

## 2. Failure Rate Analysis
Pivot the data to compute failure and success rates per payment type.

In [ ]:
pivot = df_volume.pivot_table(index='PAYMENT_TYPE', columns='STATUS', values='TXN_COUNT', fill_value=0).reset_index()
pivot['TOTAL'] = pivot.sum(axis=1, numeric_only=True)
pivot['FAILURE_RATE'] = round(pivot.get('FAILED', 0) / pivot['TOTAL'] * 100, 2)
pivot['SUCCESS_RATE'] = round(pivot.get('COMPLETED', 0) / pivot['TOTAL'] * 100, 2)

st.dataframe(pivot[['PAYMENT_TYPE', 'TOTAL', 'FAILURE_RATE', 'SUCCESS_RATE']])

## 3. Daily Failure Trends (Last 30 Days)
SQL query feeds directly into a matplotlib chart — no data export needed.

In [ ]:
df_daily = session.sql("""
    SELECT 
        PAYMENT_DATE::DATE AS DAY,
        PAYMENT_TYPE,
        COUNT(*) AS TOTAL_TXNS,
        COUNT(CASE WHEN STATUS = 'FAILED' THEN 1 END) AS FAILED_TXNS,
        ROUND(100.0 * COUNT(CASE WHEN STATUS = 'FAILED' THEN 1 END) / COUNT(*), 2) AS FAIL_RATE_PCT
    FROM PAYMENTS
    WHERE PAYMENT_DATE >= DATEADD(DAY, -30, (SELECT MAX(PAYMENT_DATE) FROM PAYMENTS))
    GROUP BY DAY, PAYMENT_TYPE
    ORDER BY DAY
""").to_pandas()

# Aggregate across payment types for overall trend
daily_agg = df_daily.groupby('DAY', as_index=False).agg(
    TOTAL_TXNS=('TOTAL_TXNS', 'sum'),
    FAILED_TXNS=('FAILED_TXNS', 'sum')
)
daily_agg['FAIL_RATE'] = round(daily_agg['FAILED_TXNS'] / daily_agg['TOTAL_TXNS'] * 100, 2)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(daily_agg['DAY'], daily_agg['FAIL_RATE'], color='#ef4444', marker='o', markersize=4, linewidth=2)
ax.fill_between(daily_agg['DAY'], daily_agg['FAIL_RATE'], alpha=0.1, color='#ef4444')
ax.set_xlabel('Date')
ax.set_ylabel('Failure Rate %')
ax.set_title('Daily Payment Failure Rate (Last 30 Days)')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
st.pyplot(fig)

## 4. Failure Rate by Payment Type
Which payment types fail the most?

In [ ]:
type_agg = df_daily.groupby('PAYMENT_TYPE', as_index=False).agg(
    TOTAL=('TOTAL_TXNS', 'sum'),
    FAILED=('FAILED_TXNS', 'sum')
)
type_agg['FAIL_RATE'] = round(type_agg['FAILED'] / type_agg['TOTAL'] * 100, 2)
type_agg = type_agg.sort_values('FAIL_RATE', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#ef4444' if r > 15 else '#29B5E8' for r in type_agg['FAIL_RATE']]
ax.bar(type_agg['PAYMENT_TYPE'], type_agg['FAIL_RATE'], color=colors, edgecolor='none')
ax.set_xlabel('Payment Type')
ax.set_ylabel('Failure Rate %')
ax.set_title('Failure Rate by Payment Type')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
st.pyplot(fig)

## 5. Processing Time: P50 vs P95
Are any payment types consistently slow?

In [ ]:
df_proc = session.sql("""
    SELECT 
        PAYMENT_TYPE,
        ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY PROCESSING_TIME_MS), 0) AS P50_MS,
        ROUND(PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY PROCESSING_TIME_MS), 0) AS P95_MS
    FROM PAYMENTS
    GROUP BY PAYMENT_TYPE
    ORDER BY P95_MS DESC
""").to_pandas()

fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(df_proc))
width = 0.35
ax.bar([i - width/2 for i in x], df_proc['P50_MS'], width, label='P50', color='#29B5E8')
ax.bar([i + width/2 for i in x], df_proc['P95_MS'], width, label='P95', color='#f59e0b')
ax.set_xticks(x)
ax.set_xticklabels(df_proc['PAYMENT_TYPE'], rotation=45, ha='right')
ax.set_ylabel('Processing Time (ms)')
ax.set_title('Processing Time: P50 vs P95 by Payment Type')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
st.pyplot(fig)

## Summary

This notebook demonstrates:
- **SQL via Snowpark** — querying Snowflake tables directly from Python
- **Pandas** — pivots, groupbys, data manipulation
- **Matplotlib charts** — operational visualisation (pre-installed, no packages needed)
- **Streamlit widgets** — `st.dataframe()` and `st.pyplot()` for rich output
- All running on **Snowflake compute** — no data leaves the platform

### Deployment
To schedule this notebook as a recurring pipeline:
1. Click **Schedule** in the notebook toolbar
2. Set frequency (e.g. daily at 06:00)
3. Assign a warehouse

The notebook becomes a governed, scheduled job — same as a Task or dbt run.